# Exercise: Gaussian Processes

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

seed = 1
torch.manual_seed(seed)
np.random.seed(seed)

# By default torch uses float32, this can lead to numerical problems for the (semi-)positive definiteness of tensors.
torch.set_default_dtype(torch.float64)

## Introduction
The goal of this exercise is to develop a deeper understanding of how kernel methods work.
We will start by implementing a simple squared exponential kernel and using it to assemble a covariance matrix.
To start as simple as possible, we will do this part of the coding in a very explicit way, with little happening on the background.

## Exercise: Implementing a squared exponential kernel

As we have seen in the lecture, the Squared Exponential is a popular choice of kernel when building GPs. It is given by the simple functional form:

$$
k(\mathbf{x},\mathbf{x}') = \sigma_f^2\exp\left(\displaystyle-\frac{1}{2\ell^2}\lVert\mathbf{x}-\mathbf{x}'\rVert^2\right).
$$

And as we can see above it has two hyperparameters: the scaling factor $\sigma_f$ and the length scale $\ell$ that controls the correlation between data points. We pass these parameters as `**hyperparams` to make it simple to switch between covariance functions with different parameters.

To start as simple as possible, we will assume a 1D input space $x$. Use the block below to implement the squared exponential kernel:

In [ ]:
# Implement the covariance function
def squared_exponential(x1, x2, **hyperparams):
    sig_f = hyperparams["sig_f"]
    length = hyperparams["length"]

    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #

    return out

Now let us set some initial values for the hyperparameters and test some intuitions from the lecture:

In [ ]:
# Set up some hyperparameters
hyperparams = {
    "sig_f": 1,
    "length": 1,
}

# Look at the kernel value between x=0 and itself
print('k(0,0) = ', squared_exponential(0,0,**hyperparams))

# Now look at the value between x=0 and x=1 and x=2.
# What happens as we move away from x=0?

# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

# Now try the same but first set the length scale to 0.1
# What can you observe?

# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

## Exercise: Assembling a GP covariance matrix

We now want to build a GP and **sample functions** from it. Assuming zero mean, we can represent our function $y$ by:

$$
y \sim \mathcal{GP}\left(y(x)\lvert 0, k(x,x')\right)
$$

In practice we will be sampling our function at a number of different values of $x$, say at  `x=np.linspace(-1,1,100)`, so at 100 equally-spaced values between $-1$ and $1$. We can then **stack our 100 $y$ values** and represent them with a Gaussian:

$$
\mathbf{y} \sim \mathcal{N}\left(\mathbf{y}\lvert\mathbf{0},\mathbf{K}\right)
$$

where $\mathbf{K}$ has size $100\times 100$ and each entry relates two values of $x$:

$$
K_{ij} = k(x_i,x_j)
$$

which means **each $x$ value is compared with every other value** forming a row of the matrix.

Use the block below to assemble your matrix and then use the `plot_covariance` function to take a look at it:

In [ ]:
def covariance_matrix(x1, x2):
    K = np.zeros((len(x1),len(x2)))
    # Assemble K below. TIP: oop over rows and columns and use the kernel you implemented
    # ---------------------- student exercise --------------------------------- #
    # YOUR CODE HERE
    # ---------------------- student exercise --------------------------------- #
    return K

def plot_covariance(cov):
    fig, ax1 = plt.subplots(1, 1, figsize=(4, 3))

    # Plot covariance matrix
    im = ax1.imshow(cov)
    cbar = plt.colorbar(im, ax=ax1, fraction=0.045, pad=0.05)
    cbar.ax.set_ylabel("$k(x,x)$", fontsize=10)
    ax1.set_title(("Example of covariance matrix"))
    ax1.set_xlabel("x", fontsize=13)
    ax1.set_ylabel("x", fontsize=13)
    ax1.grid(False)

x = np.linspace(-1,1,100)

# Use the covariance_matrix function you just coded to get K
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

plot_covariance(K)

As an extra exercise, try giving the hyperparameters different values and seeing what happens with $\mathbf{K}$.

## Sample realizations

Remember a GP is a **distribution over functions**, so let us sample some functions from our Gaussian. Each function we sample will be a vector with $100$ values, each corresponding to a different $y(x_i)$

In [ ]:
# Set kernel hyperparameters
hyperparams = {
    "sig_f": 1,
    "length": 1,
}

# Sample from the Gaussian process distribution
n_points = 100  # Number of data points in each function

n_samples = 10  # Number of GP samples

# Use your code to build a covariance matrix `K` with `n_points` points in the interval [-2,2]
# Make sure your vector with inputs is called `x`
# ---------------------- student exercise --------------------------------- #
# YOUR CODE HERE
# ---------------------- student exercise --------------------------------- #

# Draw samples from a multivariate normal function with our covariance function
samples = torch.tensor(
    np.random.multivariate_normal(mean=np.zeros(n_points), cov=K, size=n_samples)
)

# Plot the sampled functions
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
for i in range(n_samples):
    ax.plot(x, samples[i], linestyle="-", marker="o", markersize=3)
ax.set_xlabel("$x$", fontsize=13)
ax.set_ylabel("$y = f(x)$", fontsize=13)
ax.set_title(f"{n_samples} different GP realizations at {n_points} points\n")
ax.set_xlim([-2.1, 2.1])
plt.show()

Now change the kernel hyperparameters and see what happens!

## Review

This is all it takes to build a GP, it is nothing more than a Gaussian distribution. The only difference is that since we cannot really sample functions, what we can do is pick a number of points $x_i$ and sample corresponding values of $y(x_i$ that represent our function.

But this is still not a regression model, as we did not observe any data. This will be covered in the next lecture, and the exercise will be expanded to allow for that.